# Cibuco_Boriken — BirdCLEF+ 2026 Inference v3 (Ensemble)
**Team:** Cibuco_Boriken | **Ensemble:** EfficientNet-B2 + ConvNeXt-Tiny | **T=0.3** | **234 species**

Average-probability ensemble of two diverse backbones trained with Smart Crop + Secondary Labels.

In [ ]:
# ── Cell 1: Install dependencies ──
!pip install -q librosa audioread
print('Dependencies installed ✅')

In [ ]:
# ── Cell 2: Clone repo + add to path ──
!git clone https://github.com/drosadocastro-bit/cibuco-boriken
import sys
sys.path.insert(0, '/kaggle/working/cibuco-boriken')
print('Repo cloned ✅')

In [ ]:
# ── Cell 3: Imports + config ──
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as nnF
import librosa
from pathlib import Path
from tqdm import tqdm

# Paths
DATA_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
# ⬇️ UPDATE THESE to your uploaded model dataset paths on Kaggle
B2_MODEL_PATH       = Path('/kaggle/input/datasets/drakus74/efficientnet-b2-smartcrop/efficientnet_b2_SMARTCROP_BCE_20260315.pt')
CONVNEXT_MODEL_PATH = Path('/kaggle/input/datasets/drakus74/convnext-tiny-smartcrop/convnext_tiny_SMARTCROP_BCE_20260316.pt')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')

# Config — MUST match training pipeline (birdclef/config.py)
SAMPLE_RATE = 32000
WINDOW_SEC  = 5.0
N_MELS      = 128
N_FFT       = 2048
HOP_LENGTH  = 512
FMIN        = 50
FMAX        = 14000
TEMPERATURE = 0.3
BATCH_SIZE  = 32      # smaller batch — two models in memory
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

# Zero-shot prior: species in taxonomy but not in training data
ZERO_SHOT_PRIOR = 1.0 / 234

# Ensemble weights (equal by default — tune if one model is stronger)
W_B2       = 0.5
W_CONVNEXT = 0.5

print(f'Device:      {DEVICE}')
print(f'Window:      {WINDOW_SEC}s')
print(f'Mel:         n_fft={N_FFT}, hop={HOP_LENGTH}, n_mels={N_MELS}')
print(f'Temperature: {TEMPERATURE}')
print(f'Ensemble:    B2={W_B2}, ConvNeXt={W_CONVNEXT}')
print(f'Zero-shot prior: {ZERO_SHOT_PRIOR:.6f}')
print('Config ready ✅')

In [ ]:
# ── Cell 4: Load species lists ──
taxonomy_df    = pd.read_csv(DATA_DIR / 'taxonomy.csv')
SPECIES        = taxonomy_df['primary_label'].tolist()
NUM_SPECIES    = len(SPECIES)

train_df          = pd.read_csv(DATA_DIR / 'train.csv')
TRAIN_SPECIES     = sorted(train_df['primary_label'].unique().tolist())
NUM_TRAIN_SPECIES = len(TRAIN_SPECIES)

ZERO_SHOT = set(SPECIES) - set(TRAIN_SPECIES)

print(f'Taxonomy species:   {NUM_SPECIES}')
print(f'Train species:      {NUM_TRAIN_SPECIES}')
print(f'Zero-shot species:  {len(ZERO_SHOT)}')

In [ ]:
# ── Cell 5: Load BOTH models ──
from torchvision.models import efficientnet_b2, convnext_tiny

def load_model(builder_fn, classifier_setup_fn, model_path, device):
    """Load a model with torch.compile _orig_mod prefix handling."""
    model = builder_fn(weights=None)
    classifier_setup_fn(model)
    state_dict = torch.load(model_path, map_location=device, weights_only=True)
    cleaned = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(cleaned)
    model = model.to(device)
    model.eval()
    return model

# --- EfficientNet-B2 ---
def setup_b2_classifier(model):
    in_f = model.classifier[1].in_features  # 1408
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_f, NUM_TRAIN_SPECIES),
    )

model_b2 = load_model(efficientnet_b2, setup_b2_classifier, B2_MODEL_PATH, DEVICE)
print(f'EfficientNet-B2 loaded ✅  (classifier: 1408 → {NUM_TRAIN_SPECIES})')

# --- ConvNeXt-Tiny ---
def setup_convnext_classifier(model):
    in_f = model.classifier[2].in_features  # 768
    model.classifier[2] = nn.Linear(in_f, NUM_TRAIN_SPECIES)

model_cx = load_model(convnext_tiny, setup_convnext_classifier, CONVNEXT_MODEL_PATH, DEVICE)
print(f'ConvNeXt-Tiny loaded ✅  (classifier: 768 → {NUM_TRAIN_SPECIES})')

print(f'Both models on {DEVICE} ✅')

In [ ]:
# ── Cell 6: Audio + mel utilities ──
def audio_to_mel(audio, sr=SAMPLE_RATE, n_mels=N_MELS):
    """Produce mel-spectrogram with EXACT same params as training."""
    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        n_fft=N_FFT, hop_length=HOP_LENGTH,
        fmin=FMIN, fmax=FMAX
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_min, mel_max = mel_db.min(), mel_db.max()
    if mel_max - mel_min > 1e-8:
        mel_db = (mel_db - mel_min) / (mel_max - mel_min)
    else:
        mel_db = np.zeros_like(mel_db)
    return mel_db.astype(np.float32)

def mel_to_tensor(mel):
    """Convert (n_mels, time) → (3, 224, 224)."""
    t = torch.from_numpy(mel).unsqueeze(0)
    t = t.repeat(3, 1, 1)
    t = nnF.interpolate(
        t.unsqueeze(0), size=(224, 224),
        mode='bilinear', align_corners=False,
    ).squeeze(0)
    return t

@torch.no_grad()
def predict_ensemble(batch_tensor):
    """Run both models, temperature-scale, and average probabilities."""
    logits_b2 = model_b2(batch_tensor)
    logits_cx = model_cx(batch_tensor)
    probs_b2 = torch.sigmoid(logits_b2 / TEMPERATURE)
    probs_cx = torch.sigmoid(logits_cx / TEMPERATURE)
    # Weighted average of probabilities
    probs = W_B2 * probs_b2 + W_CONVNEXT * probs_cx
    return probs.cpu().numpy()

print('Audio utilities + ensemble predictor ready ✅')

In [ ]:
# ── Cell 7: Parse sample submission for exact row_ids ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'Sample submission: {len(sample_sub)} rows, {len(sample_sub.columns)} cols')
print(f'First row_id: {sample_sub["row_id"].iloc[0]}')
print(f'Last  row_id: {sample_sub["row_id"].iloc[-1]}')

def parse_row_id(row_id):
    parts = row_id.rsplit('_', 1)
    return parts[0], int(parts[1])

from collections import defaultdict
file_windows = defaultdict(list)
for row_id in sample_sub['row_id']:
    stem, end_time = parse_row_id(row_id)
    file_windows[stem].append((row_id, end_time))

print(f'Soundscape files: {len(file_windows)}')
for stem in list(file_windows.keys())[:3]:
    end_times = [et for _, et in file_windows[stem]]
    print(f'  {stem}: {len(end_times)} windows, end_times={end_times}')
print('Row IDs parsed ✅')

In [ ]:
# ── Cell 8: Run ENSEMBLE inference driven by sample_submission row_ids ──
soundscape_dir = DATA_DIR / 'test_soundscapes'
train_species_idx = {sp: i for i, sp in enumerate(TRAIN_SPECIES)}

all_rows = {}

for file_stem in tqdm(sorted(file_windows.keys()), desc='Ensemble Inference'):
    # Find the audio file
    audio_path = None
    for ext in ['.ogg', '.wav', '.flac']:
        candidate = soundscape_dir / f'{file_stem}{ext}'
        if candidate.exists():
            audio_path = candidate
            break

    if audio_path is None:
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    try:
        audio, _ = librosa.load(str(audio_path), sr=SAMPLE_RATE, mono=True)
    except Exception as e:
        print(f'Error loading {audio_path}: {e}')
        for row_id, end_time in file_windows[file_stem]:
            row = {'row_id': row_id}
            for sp in SPECIES:
                row[sp] = ZERO_SHOT_PRIOR
            all_rows[row_id] = row
        continue

    window_samples = int(WINDOW_SEC * SAMPLE_RATE)
    windows_info = file_windows[file_stem]
    tensors = []
    row_ids_order = []

    for row_id, end_time in windows_info:
        start_sample = int((end_time - WINDOW_SEC) * SAMPLE_RATE)
        end_sample   = int(end_time * SAMPLE_RATE)
        start_sample = max(0, start_sample)
        chunk = audio[start_sample:end_sample]
        if len(chunk) < window_samples:
            chunk = np.pad(chunk, (0, window_samples - len(chunk)))
        mel = audio_to_mel(chunk)
        t = mel_to_tensor(mel)
        tensors.append(t)
        row_ids_order.append(row_id)

    # Batch ensemble inference
    batch_all = torch.stack(tensors).to(DEVICE)
    probs_list = []
    for i in range(0, len(batch_all), BATCH_SIZE):
        batch = batch_all[i:i+BATCH_SIZE]
        probs_list.append(predict_ensemble(batch))
    train_probs = np.vstack(probs_list)

    for idx, row_id in enumerate(row_ids_order):
        row = {'row_id': row_id}
        for sp in SPECIES:
            if sp in train_species_idx:
                row[sp] = float(train_probs[idx, train_species_idx[sp]])
            else:
                row[sp] = ZERO_SHOT_PRIOR
        all_rows[row_id] = row

print(f'Total rows: {len(all_rows)} (expected: {len(sample_sub)}) ✅')

In [ ]:
# ── Cell 9: Generate submission.csv (exact format match) ──
rows_ordered = [all_rows[rid] for rid in sample_sub['row_id']]
submission_df = pd.DataFrame(rows_ordered)
submission_df = submission_df[sample_sub.columns]
submission_df = submission_df.fillna(ZERO_SHOT_PRIOR)
submission_df.to_csv(OUTPUT_PATH, index=False)

print(f'Submission saved ✅')
print(f'Shape: {submission_df.shape} (expected: {sample_sub.shape})')
print(f'Min prob: {submission_df.iloc[:, 1:].min().min():.6f}')
print(f'Max prob: {submission_df.iloc[:, 1:].max().max():.6f}')
print(submission_df.head(3))

In [ ]:
# ── Cell 10: Validate submission ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
our_sub    = pd.read_csv(OUTPUT_PATH)

print('=== Submission Validation ===')
print(f'Expected rows:  {len(sample_sub)}')
print(f'Our rows:       {len(our_sub)}')
print(f'Expected cols:  {len(sample_sub.columns)}')
print(f'Our cols:       {len(our_sub.columns)}')
print(f'Columns match:  {set(sample_sub.columns) == set(our_sub.columns)}')
print(f'Row IDs match:  {set(sample_sub["row_id"]) == set(our_sub["row_id"])}')
print(f'No NaN:         {our_sub.isna().sum().sum() == 0}')
print(f'No zeros:       {(our_sub.iloc[:, 1:] == 0).sum().sum() == 0}')
print()
if set(sample_sub.columns) == set(our_sub.columns):
    print('✅ ENSEMBLE SUBMISSION VALID — ready to submit!')
else:
    missing = set(sample_sub.columns) - set(our_sub.columns)
    extra   = set(our_sub.columns) - set(sample_sub.columns)
    if missing: print(f'⚠️  Missing columns: {missing}')
    if extra:   print(f'⚠️  Extra columns: {extra}')